# 🛡️ Huấn Luyện Bộ Lọc Xác Thực Wake Word (Alexa Verifier)

Notebook này sử dụng tập dữ liệu **dataset_wakeword.tar.gz** (gồm 30 mẫu bạn gọi *"Alexa"* và 50 mẫu nhạc/tạp âm) để huấn luyện một bộ lọc nhị phân (Binary Classifier Verifier).

### Mục tiêu:
- Ngăn chặn **100% tình trạng tự động kích hoạt nhầm** khi loa laptop bật nhạc to hoặc có tiếng ồn xung quanh.
- Sau khi train (~30 giây), tự động tải về file **`alexa_verifier.pkl`** để đặt vào laptop.

## 1. Cài đặt thư viện cần thiết

In [ ]:
!pip install -q openwakeword onnx scikit-learn soundfile numpy scipy librosa

## 2. Giải nén dataset_wakeword.tar.gz
> **Lưu ý**: Hãy kéo thả file `dataset_wakeword.tar.gz` từ máy tính vào thanh Files bên trái của Colab trước khi chạy ô này!

In [ ]:
import os
import glob

if os.path.exists("dataset_wakeword.tar.gz"):
    !tar -xzf dataset_wakeword.tar.gz
    print("✅ Đã giải nén dataset_wakeword.tar.gz thành công!")
else:
    print("⚠️ Chưa tìm thấy file dataset_wakeword.tar.gz. Hãy tải file lên Colab trước!")

pos_files = glob.glob("training/wakeword/dataset/positive/*.wav")
neg_files = glob.glob("training/wakeword/dataset/negative/*.wav")
print(f"🎯 Số mẫu Dương (Positive - Giọng bạn): {len(pos_files)}")
print(f"🎯 Số mẫu Âm (Negative - Nhạc/Tạp âm): {len(neg_files)}")

## 3. Trích xuất đặc trưng Embeddings bằng openWakeWord

In [ ]:
import numpy as np
import soundfile as sf
import openwakeword
from openwakeword.model import Model
import pickle

# Nạp mô hình Alexa chuẩn
alexa_models = [p for p in openwakeword.get_pretrained_model_paths() if "alexa" in p]
base_model = Model(wakeword_model_paths=alexa_models)
model_name = list(base_model.models.keys())[0]

def extract_features_from_audio(file_path):
    data, sr = sf.read(file_path)
    if sr != 16000:
        import librosa
        data = librosa.resample(data, orig_sr=sr, target_sr=16000)
    if data.dtype != np.int16:
        data = (data * 32767).astype(np.int16)
    
    features = []
    base_model.reset()
    for i in range(0, len(data) - 1280, 1280):
        chunk = data[i:i+1280]
        base_model.predict(chunk)
        feat = base_model.preprocessor.get_features(base_model.model_inputs[model_name])
        features.append(feat.flatten())
    return features

X = []
y = []

print("⏳ Đang trích xuất đặc trưng mẫu Dương (Alexa của bạn)...")
for f in pos_files:
    for feat in extract_features_from_audio(f):
        X.append(feat)
        y.append(1)

print("⏳ Đang trích xuất đặc trưng mẫu Âm (Âm nhạc & Tiếng ồn)...)")
for f in neg_files:
    for feat in extract_features_from_audio(f):
        X.append(feat)
        y.append(0)

X = np.array(X)
y = np.array(y)
print(f"✅ Trích xuất hoàn tất! Tổng số khung hình: {len(X)} | Dương: {np.sum(y == 1)} | Âm: {np.sum(y == 0)}")

## 4. Huấn luyện Bộ lọc Verifier & Xuất file alexa_verifier.pkl

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0)
clf.fit(X, y)

preds = clf.predict(X)
print(classification_report(y, preds, target_names=["Nhạc/Tạp âm (Âm)", "Alexa của bạn (Dương)"]))

# Xuất file
with open("alexa_verifier.pkl", "wb") as f:
    pickle.dump(clf, f)

print("=" * 60)
print("🎉 ĐÃ HUẤN LUYỆN THÀNH CÔNG!")
print("File 'alexa_verifier.pkl' đã được tạo.")
print("=" * 60)

# Tự động tải về máy
from google.colab import files
files.download("alexa_verifier.pkl")